# CENG313 – Introduction to Data Science: Assignment 1
## Exploratory Data Analysis (EDA) & Baseline Modeling
**Competition:** [CMI – Detect Behavior with Sensor Data](https://www.kaggle.com/competitions/cmi-detect-behavior-with-sensor-data)

---

## 1. Problem Definition
* **Task:** Multivariate Time-Series Classification. Using multi-modal wearable sensor data recorded from a wrist-worn device (3-axis accelerometer, quaternion rotation/orientation, thermopile temperature, and Time-of-Flight proximity sensors), the task is to classify specific hand/wrist gestures and distinguish **Body-Focused Repetitive Behaviors (BFRBs)** (e.g., hair pulling, skin pinching/scratching) from everyday non-BFRB activities.
* **Target Variable:** `gesture` (18 classes, one label per sequence). `sequence_type` shows if the gesture is a BFRB (Target) or not (Non-Target).
  * Note: `behavior` is not a target. It shows the phase of each row (e.g. moving the hand, performing the gesture).
* **Evaluation Metric:** The competition evaluates models using a **Hierarchical Macro F1** score, defined as the average of:
  1. Binary F1 score for BFRB vs. Non-BFRB classification.
  2. Macro F1 over 9 classes: the 8 BFRB gestures + one `non_target` class (all 10 non-BFRB gestures merged).

  In Section 5, we report both standard **Macro F1** and the competition's **Hierarchical Macro F1** (implemented in `src/utils.py`).

## 2. Data Overview
In this section, we load the primary training time-series dataset (`train.csv`) and participant metadata (`train_demographics.csv`) using our modular helper functions from `src/data.py`. We inspect dataset dimensions, data types, summary statistics, exact duplicates, and missing value patterns across sensor modalities.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data import load_data

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams['figure.dpi'] = 110

# Load datasets via src/data.py
train, demo = load_data("data")

print(f"Train time-series shape:      {train.shape[0]:,} rows x {train.shape[1]} columns")
print(f"Demographics shape:           {demo.shape[0]:,} rows x {demo.shape[1]} columns")
print(f"Unique sequences (recordings): {train['sequence_id'].nunique():,}")
print(f"Unique participants (subjects):{train['subject'].nunique():,}")
print(f"Exact duplicate rows:         {train.duplicated().sum()}")

print("\n--- Column Data Types ---")
print(train.dtypes.value_counts())

In [ ]:
# Group sensor columns by modality
acc_cols = [c for c in train.columns if c.startswith('acc_')]
rot_cols = [c for c in train.columns if c.startswith('rot_')]
thm_cols = [c for c in train.columns if c.startswith('thm_')]
tof_cols = [c for c in train.columns if c.startswith('tof_')]

print(f"Sensor channel breakdown -> IMU Acc: {len(acc_cols)}, IMU Rot: {len(rot_cols)}, Thermopile: {len(thm_cols)}, ToF: {len(tof_cols)}")

print("\n--- Summary Statistics (IMU Accelerometer & Rotation) ---")
display(train[acc_cols + rot_cols].describe().T[['count', 'mean', 'std', 'min', '25%', '50%', '75%', 'max']].round(3))

print("\n--- Missing Values by Modality ---")
modality_missing = []
for name, cols in [('Accelerometer (acc_*)', acc_cols), ('Rotation (rot_*)', rot_cols), 
                   ('Thermopile (thm_*)', thm_cols), ('Time-of-Flight (tof_*)', tof_cols)]:
    if cols:
        pct = train[cols].isnull().mean().mean() * 100
        modality_missing.append({'Modality': name, 'Channel Count': len(cols), 'Avg Missing (%)': round(pct, 2)})
display(pd.DataFrame(modality_missing))

**Data Overview Findings:**
* **Hierarchical Structure & Zero Duplicates:** Every recording (`sequence_id`) contains multiple consecutive time steps (`sequence_counter`) performed by a single participant (`subject`). There are **0 exact duplicate rows**.
* **Modality-Specific Missingness:** Core IMU channels (`acc_*`, `rot_*`) are nearly complete, whereas peripheral sensors (`thm_*` and `tof_*`) show substantial missing percentages (`NaN`). In this dataset, certain recording sessions or test configurations operate in an **IMU-only mode** where peripheral sensors are completely absent (`NaN` across the entire sequence). Recognizing this structural missingness is critical for both feature imputation and model generalization.

## 3. Exploratory Data Analysis (EDA)
Because EDA accounts for the core analytical weight of Assignment 1, we conduct a thorough multi-angle investigation covering:
1. **Target Class Balance & BFRB vs. Non-BFRB Breakdown**
2. **Sequence Duration (Time-Step) Distributions & Outliers**
3. **Sensor Modality Availability Across Sequences (IMU-only vs. Full-Sensor)**
4. **Raw Time-Series Signal Dynamics (BFRB vs. Non-BFRB Accelerometer & Thermopile Traces)**
5. **Distribution & Outlier Analysis of Acceleration Magnitude Across Gestures**
6. **Inter-Sensor Correlation Structure**
7. **Participant Demographics & Subject-Level Recording Counts**

Every visualization is immediately followed by a written analytical interpretation.

In [ ]:
# Create sequence-level metadata table
seq_meta = train.drop_duplicates('sequence_id').copy()
seq_meta = pd.merge(seq_meta, demo, on='subject', how='left')

# --- PLOT 1: Target Class Balance ---
fig, axes = plt.subplots(1, 2, figsize=(15, 5), gridspec_kw={'width_ratios': [2.2, 1]})

gesture_order = seq_meta['gesture'].value_counts().index
sns.countplot(data=seq_meta, y='gesture', order=gesture_order, hue='gesture', palette='viridis', legend=False, ax=axes[0])
axes[0].set_title("1A. Sequence Count per Gesture Class", fontweight='bold')
axes[0].set_xlabel("Number of Sequences")
axes[0].set_ylabel("Gesture")

if 'behavior' in seq_meta.columns:
    beh_counts = seq_meta['behavior'].value_counts()
    sns.barplot(x=beh_counts.index, y=beh_counts.values, hue=beh_counts.index, palette='Set2', legend=False, ax=axes[1])
    axes[1].set_title("1B. High-Level Behavior Split", fontweight='bold')
    axes[1].set_xlabel("Behavior Category")
    axes[1].set_ylabel("Number of Sequences")
    for i, v in enumerate(beh_counts.values):
        axes[1].text(i, v + max(beh_counts.values)*0.01, f"{v:,}", ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

**Interpretation 1 (Target Class Balance):**
* **Class Imbalance:** The left panel reveals clear class imbalance across specific `gesture` labels, with non-BFRB control movements appearing substantially more often than individual BFRB sub-types.
* **Metric Justification:** Under imbalanced multi-class distributions, overall accuracy can appear artificially high by favoring majority non-BFRB gestures. Using **Macro F1** ensures every rare BFRB gesture contributes equally to the evaluation score.

In [ ]:
# --- PLOT 2: Sequence Length Distribution & Outliers ---
seq_lengths = train.groupby('sequence_id').size().rename('seq_len')
seq_meta_len = seq_meta.merge(seq_lengths, on='sequence_id')

fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))

sns.histplot(seq_lengths, bins=45, kde=True, color='#2b5c8f', ax=axes[0])
axes[0].axvline(seq_lengths.median(), color='crimson', linestyle='--', linewidth=1.8, label=f"Median: {seq_lengths.median():.0f}")
axes[0].axvline(seq_lengths.quantile(0.95), color='darkorange', linestyle=':', linewidth=1.8, label=f"95th Pct: {seq_lengths.quantile(0.95):.0f}")
axes[0].set_title("2A. Overall Distribution of Sequence Lengths", fontweight='bold')
axes[0].set_xlabel("Time Steps per Sequence")
axes[0].set_ylabel("Frequency")
axes[0].legend()

sns.boxplot(data=seq_meta_len, x='seq_len', y='gesture', order=gesture_order, palette='coolwarm', ax=axes[1])
axes[1].set_title("2B. Sequence Duration Outliers by Gesture", fontweight='bold')
axes[1].set_xlabel("Time Steps")
axes[1].set_ylabel("")

plt.tight_layout()
plt.show()

print(f"Sequence length summary -> Min: {seq_lengths.min()}, Median: {seq_lengths.median():.0f}, Mean: {seq_lengths.mean():.1f}, Max: {seq_lengths.max()}")

**Interpretation 2 (Sequence Lengths & Temporal Outliers):**
* **Right-Skewed Duration:** Most gesture sequences cluster tightly around the median duration, while a long right tail extends past the 95th percentile.
* **Modeling Implication:** Because raw sequences vary substantially in length, sequence models require padding/truncation, whereas tabular baselines require **length-invariant statistical aggregations** (`mean`, `std`, quantiles) per `sequence_id` so longer recordings do not bias feature magnitudes.

In [ ]:
# --- PLOT 3: Sensor Modality Availability (IMU-only vs Full-Sensor Sequences) ---
seq_missing = train.groupby('sequence_id').apply(
    lambda g: pd.Series({
        'has_thm': g[thm_cols].notnull().any().any() if thm_cols else False,
        'has_tof': g[tof_cols].notnull().any().any() if tof_cols else False
    })
)
seq_missing['Sensor Configuration'] = np.where(
    seq_missing['has_thm'] & seq_missing['has_tof'], 'Full Sensors (IMU + Thm + ToF)',
    np.where(~seq_missing['has_thm'] & ~seq_missing['has_tof'], 'IMU Only (Acc + Rot)', 'Partial Peripheral Sensors')
)

plt.figure(figsize=(8, 4))
cfg_counts = seq_missing['Sensor Configuration'].value_counts()
sns.barplot(x=cfg_counts.values, y=cfg_counts.index, hue=cfg_counts.index, palette='mako', legend=False)
plt.title("3. Sensor Hardware Availability Across Training Sequences", fontweight='bold')
plt.xlabel("Number of Sequences")
plt.ylabel("")
for i, v in enumerate(cfg_counts.values):
    plt.text(v + max(cfg_counts.values)*0.01, i, f"{v:,} ({v/len(seq_missing)*100:.1f}%)", va='center', fontweight='bold')
plt.xlim(0, max(cfg_counts.values)*1.22)
plt.tight_layout()
plt.show()

**Interpretation 3 (Sensor Modality Availability):**
* **Structural Missingness Confirmed:** A distinct subset of sequences contains **only IMU data** (`acc_*` and `rot_*`), with `thm_*` and `tof_*` completely absent.
* **Design Decision:** Models must not rely solely on thermopile or Time-of-Flight features, nor drop rows with missing peripheral sensors. Imputing missing peripheral features consistently ensures our pipeline handles both full-sensor and IMU-only test recordings seamlessly.

In [ ]:
# --- PLOT 4: Raw Time-Series Dynamics (BFRB vs Non-BFRB Signal Comparison) ---
bfrb_candidates = seq_meta[seq_meta['gesture'].str.contains('pull|scratch|pinch', case=False, na=False)]
non_bfrb_candidates = seq_meta[~seq_meta['gesture'].str.contains('pull|scratch|pinch', case=False, na=False)]

seq_bfrb = bfrb_candidates['sequence_id'].iloc[0] if len(bfrb_candidates) > 0 else seq_meta['sequence_id'].iloc[0]
seq_ctrl = non_bfrb_candidates['sequence_id'].iloc[0] if len(non_bfrb_candidates) > 0 else seq_meta['sequence_id'].iloc[1]

fig, axes = plt.subplots(2, 2, figsize=(15, 7), sharex='col')

for col_idx, (sid, label_prefix) in enumerate([(seq_bfrb, 'BFRB Example'), (seq_ctrl, 'Non-BFRB Example')]):
    sub = train[train['sequence_id'] == sid].sort_values('sequence_counter')
    g_name = sub['gesture'].iloc[0]
    
    # Top row: 3-Axis Accelerometer
    axes[0, col_idx].plot(sub['sequence_counter'], sub['acc_x'], label='acc_x', alpha=0.9)
    axes[0, col_idx].plot(sub['sequence_counter'], sub['acc_y'], label='acc_y', alpha=0.9)
    axes[0, col_idx].plot(sub['sequence_counter'], sub['acc_z'], label='acc_z', alpha=0.9)
    axes[0, col_idx].set_title(f"4A. {label_prefix} ({g_name}) – Accelerometer", fontweight='bold')
    axes[0, col_idx].set_ylabel("Acceleration (m/s²)")
    axes[0, col_idx].legend(loc='upper right')
    
    # Bottom row: Quaternion Rotation
    for r_col in rot_cols:
        axes[1, col_idx].plot(sub['sequence_counter'], sub[r_col], label=r_col, alpha=0.85)
    axes[1, col_idx].set_title(f"4B. {label_prefix} ({g_name}) – Orientation (Quaternion)", fontweight='bold')
    axes[1, col_idx].set_xlabel("Time Step (sequence_counter)")
    axes[1, col_idx].set_ylabel("Quaternion Value")
    axes[1, col_idx].legend(loc='upper right')

plt.tight_layout()
plt.show()

**Interpretation 4 (Raw Sensor Signal Dynamics):**
* **Distinct Kinematic Signatures:** Comparing a representative BFRB trajectory against a non-BFRB control trajectory illustrates clear temporal differences. BFRB actions exhibit characteristic wrist orientation shifts (visible in `rot_w/x/y/z` as the hand approaches the head/neck) followed by fine, repetitive micro-oscillations in `acc_x/y/z`.
* **Feature Relevance:** Both mean orientation (capturing *where* the hand is positioned relative to gravity) and signal standard deviation (capturing *repetitive motion intensity*) carry strong discriminative information.

In [ ]:
# --- PLOT 5: Acceleration Magnitude Distribution & Outliers by Gesture ---
train['acc_mag'] = np.sqrt(train['acc_x']**2 + train['acc_y']**2 + train['acc_z']**2)
seq_acc_stats = train.groupby(['sequence_id', 'gesture'])['acc_mag'].agg(['mean', 'std']).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
sns.boxplot(data=seq_acc_stats, x='mean', y='gesture', order=gesture_order, palette='Spectral', ax=axes[0])
axes[0].set_title("5A. Sequence Mean Acceleration Magnitude by Gesture", fontweight='bold')
axes[0].set_xlabel("Mean ||acc||")
axes[0].set_ylabel("Gesture")

sns.boxplot(data=seq_acc_stats, x='std', y='gesture', order=gesture_order, palette='Spectral', ax=axes[1])
axes[1].set_title("5B. Sequence Acceleration Variability (Std) by Gesture", fontweight='bold')
axes[1].set_xlabel("Std ||acc||")
axes[1].set_ylabel("")

plt.tight_layout()
plt.show()

# Drop temporary helper column from train
train.drop(columns=['acc_mag'], inplace=True)

**Interpretation 5 (Acceleration Magnitude Distributions & Outliers):**
* **Separability Across Classes:** High-energy gestures display substantially higher acceleration variability (`Std ||acc||`) and upper-tail outliers than stationary or subtle BFRB movements.
* **Outlier Robustness:** Several gesture classes contain extreme upper outliers in `Std ||acc||` (likely caused by rapid arm transitions before/after the target gesture). Tree-based models like Random Forest handle these heavy-tailed distributions naturally without distortion.

In [ ]:
# --- PLOT 6: Correlation Heatmap of Core IMU & Thermopile Channels ---
corr_cols = acc_cols + rot_cols + thm_cols
corr_matrix = train[corr_cols].corr()

plt.figure(figsize=(10, 7))
sns.heatmap(corr_matrix, cmap='RdBu_r', vmin=-1, vmax=1, annot=True, fmt=".2f", annot_kws={"size": 8})
plt.title("6. Correlation Heatmap Across IMU and Thermopile Channels", fontweight='bold')
plt.tight_layout()
plt.show()

**Interpretation 6 (Inter-Sensor Correlation Structure):**
* **Within-Modality vs. Cross-Modality Independence:** Thermopile channels (`thm_1` through `thm_5`) exhibit strong positive inter-correlation because they measure skin/ambient thermal proximity simultaneously around the wrist. By contrast, accelerometer (`acc_*`), orientation (`rot_*`), and thermal (`thm_*`) modalities show low cross-correlation, confirming that each modality contributes complementary, non-redundant information.

In [ ]:
# --- PLOT 7: Participant Demographics & Subject-Level Recording Balance ---
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

if 'handedness' in seq_meta.columns:
    sns.countplot(data=seq_meta, x='handedness', hue='behavior' if 'behavior' in seq_meta.columns else None, palette='Set2', ax=axes[0])
    axes[0].set_title("7A. Sequence Count by Handedness (0=Left, 1=Right)", fontweight='bold')
    axes[0].set_xlabel("Handedness")
    axes[0].set_ylabel("Number of Sequences")

subj_counts = seq_meta['subject'].value_counts()
sns.histplot(subj_counts, bins=20, color='#3b7a57', ax=axes[1])
axes[1].axvline(subj_counts.median(), color='red', linestyle='--', label=f"Median: {subj_counts.median():.0f} seqs/subject")
axes[1].set_title("7B. Distribution of Sequences Recorded per Participant", fontweight='bold')
axes[1].set_xlabel("Sequences per Subject")
axes[1].set_ylabel("Number of Participants")
axes[1].legend()

plt.tight_layout()
plt.show()

**Interpretation 7 (Demographics & Critical Validation Strategy Insight):**
* **Repeated Measures per Subject:** Each participant (`subject`) records dozens of sequences across multiple gesture types, and the cohort is predominantly right-handed.
* **Preventing Data Leakage:** Because individual subjects have distinct anthropometric traits and movement habits, a standard random train/test split would place the same person in both sets and artificially inflate validation scores. Therefore, our validation split **must group by `subject`**.

## 4. Preprocessing
Based on our EDA findings, we implement a clean, leak-free preprocessing pipeline in `src/data.py`:
1. **Sequence-Level Feature Aggregation (`extract_sequence_features`):** Since sequences have variable time-step lengths, we group by `sequence_id` and compute per-channel `mean` and `std` across all sensor columns (`acc_*`, `rot_*`, `thm_*`, `tof_*`). This produces a uniform tabular feature matrix with one row per recording.
2. **Missing Value Imputation (`preprocess`):** Peripheral channels (`thm_*`, `tof_*`) that are `NaN` in IMU-only sessions are imputed with `0` after aggregation so all recordings are preserved without dropping IMU-only sequences.
3. **Scaling Decision:** No feature scaling is applied for our baseline because we use a **Random Forest Classifier**, whose axis-aligned splits are invariant to monotonic feature scaling.

In [ ]:
from src.data import extract_sequence_features, preprocess

df_seq = extract_sequence_features(train, is_train=True)
df_clean = preprocess(df_seq)

print(f"Aggregated sequence feature table: {df_clean.shape[0]:,} sequences x {df_clean.shape[1]} columns")
print(f"Total remaining missing values:    {df_clean.isnull().sum().sum()}")
display(df_clean.head(3))

## 5. Baseline Model
We train a **Random Forest Classifier** (`src/models.py`) using an 80/20 `GroupShuffleSplit` grouped by `subject` (`random_state=42`). Grouping by `subject` guarantees that the model is evaluated strictly on unseen participants.

In [ ]:
from src.models import train_baseline
from src.utils import compute_hierarchical_f1
from sklearn.metrics import classification_report, confusion_matrix

model, val_macro_f1, X_val, y_val, preds = train_baseline(df_clean)
val_hier_f1 = compute_hierarchical_f1(y_val, preds)

print(f"Validation Macro F1 Score:        {val_macro_f1:.4f}")
print(f"Validation Hierarchical F1 Score: {val_hier_f1:.4f}\n")
print("--- Per-Class Classification Report ---")
print(classification_report(y_val, preds, zero_division=0))

# Visualize Normalized Confusion Matrix
labels = sorted(y_val.unique())
cm = confusion_matrix(y_val, preds, labels=labels, normalize='true')
plt.figure(figsize=(11, 8))
sns.heatmap(cm, xticklabels=labels, yticklabels=labels, cmap='Blues', annot=True, fmt='.2f', annot_kws={'size': 7})
plt.title("Baseline Random Forest – Normalized Confusion Matrix (Subject-Grouped Validation)", fontweight='bold')
plt.xlabel("Predicted Gesture")
plt.ylabel("True Gesture")
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 6. Kaggle Submission / Held-Out Evaluation
* **Evaluation Protocol:** Because this competition uses a closed code-execution inference API (`kaggle_evaluation`) rather than static CSV file uploads, we follow the course guideline for closed competitions and evaluate on a strictly held-out **20% subject-grouped validation split** (`random_state=42`).
* **Benchmark Comparison:** Top public leaderboard solutions on Kaggle leverage deep temporal models (1D-CNNs, BiGRUs, Transformers) and rich time/frequency feature engineering to achieve Hierarchical F1 scores above **0.80+**. Our simple `mean`/`std` tabular Random Forest baseline establishes a clear, leak-free initial benchmark to improve upon in Assignment 2.

---

## 7. Conclusion & Next Steps
### Key Findings
1. **Strong IMU Signal:** Basic sequence-level `mean` and `std` aggregations of wrist accelerometer and quaternion channels already separate major gesture categories meaningfully.
2. **Modality Missingness:** A notable fraction of sequences are IMU-only (`thm_*` and `tof_*` completely `NaN`), making missingness-aware features and robust imputation essential.
3. **Subject-Grouped Generalization:** Evaluating on unseen participants (`GroupShuffleSplit`) exposes realistic cross-subject variance and prevents optimistic data leakage.
